#  Advanced CIFAR-10 Classification Analysis

---

##  QUICK NAVIGATION
- **[DATA EXPLORATION](#data-exploration)**: Lines ~50-100
- **[ADVANCED CNN](#advanced-cnn)**: Lines ~100-180
- **[TRAINING](#training)**: Lines ~180-220
- **[EVALUATION](#evaluation)**: Lines ~220+

---

This notebook builds an **advanced CNN** for CIFAR-10 with:
- Deeper architecture
- Data augmentation
- Learning rate scheduling
- Early stopping


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

---

<a id='data-exploration'></a>
#  LOAD AND EXPLORE DATA

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training images: {x_train.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test images: {x_test.shape}")
print(f"Test labels: {y_test.shape}")

x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

## CLASS DISTRIBUTION

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
train_counts = np.bincount(y_train.flatten())
plt.bar(class_names, train_counts, color='skyblue', edgecolor='black')
plt.title('Training Set Class Distribution', fontsize=12)
plt.xticks(rotation=45)
plt.ylabel('Count')

plt.subplot(1, 2, 2)
test_counts = np.bincount(y_test.flatten())
plt.bar(class_names, test_counts, color='lightcoral', edgecolor='black')
plt.title('Test Set Class Distribution', fontsize=12)
plt.xticks(rotation=45)
plt.ylabel('Count')

plt.tight_layout()
plt.show()

---

<a id='advanced-cnn'></a>
# BUILD ADVANCED CNN

This CNN has:
- 3 convolutional blocks
- Batch normalization after each layer
- Progressive dropout rates
- 512-unit dense layer before output

In [ ]:
def build_advanced_cnn():
    model = models.Sequential([
        # Block 1
        layers.Conv2D(64, (3, 3), padding='same', activation='relu', input_shape=(32, 32, 3)),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Block 2
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),

        # Block 3
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3, 3), padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.5),

        # Classifier head
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

advanced_cnn = build_advanced_cnn()
print("="*60)
print("ADVANCED CNN ARCHITECTURE")
print("="*60)
advanced_cnn.summary()

---

# SET UP TRAINING CALLBACKS

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint = callbacks.ModelCheckpoint(
    'best_advanced_cnn.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_list = [early_stopping, lr_scheduler, checkpoint]

---

#  DATA AUGMENTATION

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(height_factor=0.1, width_factor=0.1),
    layers.RandomContrast(factor=0.1)
])

# Visualize augmented images
plt.figure(figsize=(10, 10))
for i in range(9):
    augmented_image = data_augmentation(tf.expand_dims(x_train_norm[0], 0))
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(augmented_image[0])
    plt.title(f"Augmented #{i+1}", fontsize=10)
    plt.axis("off")
plt.suptitle("Data Augmentation Examples", fontsize=14)
plt.tight_layout()
plt.show()

---

<a id='training'></a>
# COMPILE AND TRAIN THE MODEL

In [ ]:
advanced_cnn.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("="*60)
print("STARTING MODEL TRAINING")
print("="*60)

history = advanced_cnn.fit(
    data_augmentation(x_train_norm, training=True),
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.1,
    callbacks=callbacks_list
)

---

<a id='evaluation'></a>
# EVALUATE ON TEST SET

In [ ]:
test_loss, test_acc = advanced_cnn.evaluate(x_test_norm, y_test, verbose=0)
print(f"\n{'='*60}")
print(f"FINAL TEST RESULTS")
print(f"{'='*60}")
print(f"📊 Test Accuracy: {test_acc:.4f}")
print(f"📊 Test Loss: {test_loss:.4f}")

y_pred = np.argmax(advanced_cnn.predict(x_test_norm), axis=1)
y_true = y_test.flatten()

##  PLOT TRAINING HISTORY

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.legend(fontsize=12)
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## CONFUSION MATRIX AND CLASSIFICATION REPORT

In [ ]:
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##  VISUALIZE MISCLASSIFIED EXAMPLES

In [ ]:
misclassified_idx = np.where(y_pred != y_true)[0]

plt.figure(figsize=(15, 8))
for i in range(min(15, len(misclassified_idx))):
    idx = misclassified_idx[i]
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[idx])
    plt.title(f"True: {class_names[y_true[idx]]}\nPred: {class_names[y_pred[idx]]}", 
              fontsize=9, color='red')
    plt.axis('off')

plt.suptitle('Misclassified Examples', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()